##### 5-Fold Cross-Validation
5-fold cross Validation of the U-Net
folds spilt by city, to avoid near identical frames being in both train and validation 

In [ ]:
# 5 folds for cross-validation, built from the 15 train + 3 val cities (2,939 images).
# The 3 test cities (dusseldorf, jena, stuttgart) are left out so they stay unseen.
# Whole cities are kept in one fold because frames from the same city look very similar;
# splitting a city across train and validation would leak data and inflate the scores.
# Cities are grouped so each fold has a similar number of images .

FOLDS = {
    1: ["darmstadt", "strasbourg", "zurich"],
    2: ["hamburg", "hanover", "ulm", "lindau"],
    3: ["cologne", "weimar", "frankfurt"],
    4: ["aachen", "bremen", "erfurt"],
    5: ["bochum", "krefeld", "monchengladbach", "tubingen", "munster"],
}

all_cities = []
for fold, cities in FOLDS.items():
    print(f"Fold {fold}: {len(cities)} cities")
    all_cities.extend(cities)

print("Total cities:", len(all_cities))
print("Unique cities:", len(set(all_cities)))

In [ ]:
import os
import sys
from pathlib import Path
sys.path.insert(0, os.path.abspath("..")) # for src imports

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda" # enable Automatic Mixed Precision (AMP) only when using cuda GPU
scaler = torch.amp.GradScaler("cuda", enabled=use_amp) # gradient scaler to help prevent numerical underflow during mixed-precision training
torch.backends.cudnn.benchmark = True # selects efficient algorithms for convolution operations when input sizes remain consistent

print("PyTorch version:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("   GPU:", torch.cuda.get_device_name(0))
    print("   CUDA version:", torch.version.cuda)

In [ ]:
NUM_CLASSES = 19
IGNORE_INDEX = 255

DATASET_DIR = "../data/dataset" # train / val / test
MODELS_DIR = "../models" # model save
EDA_CLASS_DIST = "../eda/tables/split_class_distribution.csv"

In [ ]:
from pathlib import Path

SOURCE_DIRS = [Path(DATASET_DIR) / "train", Path(DATASET_DIR) / "val"]
CV_DIR = Path("../data/cv")

for fold, val_cities in FOLDS.items():
    fold_dir = CV_DIR / f"fold_{fold}"
    (fold_dir / "train").mkdir(parents=True, exist_ok=True)
    (fold_dir / "val").mkdir(parents=True, exist_ok=True)

    for source_dir in SOURCE_DIRS:
        for file in os.listdir(source_dir):
            city = file.split("_")[0]
            split = "val" if city in val_cities else "train"
            destination = fold_dir / split / file
            if not destination.exists():
                os.link(source_dir / file, destination)

print("Fold folders created")

In [ ]:
for fold in FOLDS:
    for split in ["train", "val"]:
        count = len(list((CV_DIR / f"fold_{fold}" / split).glob("*_leftImg8bit.png")))
        print(f"Fold {fold} {split}: {count}")

In [ ]:
VERSION = "cv"


NATIVE_SIZE = (1024, 2048) # original resolution
CROP_SIZE = None    # random crop size | None = no crops
CROP_SCALE = (0.5, 2.0)    # random scale factor range for crops
TRAIN_SIZE = (256,512)          # resize if no crops | None = native size
FLIP = True                # random horizontal flip

INPUT_SIZE = CROP_SIZE or TRAIN_SIZE or NATIVE_SIZE # training samples size

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 8
#NUM_WORKERS = min(12, os.cpu_count() or 1)
NUM_WORKERS = 4
print("Dataloader workers:", NUM_WORKERS)

# model
BASE = 32
# criterion
CE_COEFF = 1.0 # weight cross entropy loss
LOVASZ_COEFF = 0.5 # weight lovasz loss
WEIGHT_EXPONENT = 0.5 # class imbalance weights: 0 = no weights, 0.5 = sqrt
# optimizer
MAX_LEARNING_RATE = 1e-4 # start at this rate
MIN_LEARNING_RATE = 1e-6 # slowly reduce to this rate
WEIGHT_DECAY = 1e-4
# training
EPOCHS = 40

In [ ]:
VERSION_SIZE = f"{VERSION}-{INPUT_SIZE[0]}-{INPUT_SIZE[1]}"

MODEL_DIR = Path(MODELS_DIR)/VERSION_SIZE
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(INPUT_SIZE, MODEL_DIR)


Dataloaders

In [ ]:
from torch.utils.data import DataLoader
from src.dataset import CityscapesDataset

def make_loaders(fold_dir): #Makes the train and validation loaders for one fold
    train_ds = CityscapesDataset(fold_dir / "train", crop_size=CROP_SIZE, crop_scale=CROP_SCALE, resize=TRAIN_SIZE, flip=FLIP)#Training images resized to 256x512 with the random horizontal flips
    valid_ds = CityscapesDataset(fold_dir / "val", resize=TRAIN_SIZE) #validation is resized to the same size as training so the model is judged at the size it was trained on

    train_loader = DataLoader(
        train_ds, 
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True, #A different image oreders every epoch, so the model doesnt learn anything from the order 
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
    )

    valid_loader = DataLoader(
        valid_ds,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False, #order doesnt matter for measuring performance 
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
    )

    return train_loader, valid_loader


Create model

In [ ]:
from src.model import UNet
model = UNet(in_channels=3, num_classes=NUM_CLASSES, base=BASE).to(device)

# memory estimate does not account for AMP
from torchinfo import summary
summary(model, input_size=(TRAIN_BATCH_SIZE, 3, *INPUT_SIZE))

Check formats

In [ ]:
# grab one bactch to inspect shape
# [B = Batch Size, 3 = RGB, H = Height, W = Width]
train_loader, valid_loader = make_loaders(CV_DIR / "fold_1")
print("train images:", len(train_loader.dataset), "| val images:", len(valid_loader.dataset))
imgs, masks = next(iter(train_loader))
print("images:", imgs.shape, imgs.dtype)    # should be [B,3,H,W] float32
print("masks :", masks.shape, masks.dtype)  # should be [B,H,W] int64

# run model with no gradient (reduces memory usage) to inspect output shape
# [B = Batch Size, 19 = Classes, H = Height, W = Width]
# with torch.no_grad():
#     out = model(imgs.to(next(model.parameters()).device))
# print("logits:", out.shape)  # should be [B,19,H,W]

Train model

loop:
- training = forward + backward passes over training set
- evaluate = forward pass over validation set (metrics + validation loss)

In [ ]:
import time
import json
import torch
import torch.nn as nn
import pandas as pd
from tqdm import tqdm
from segmentation_models_pytorch.losses import LovaszLoss
from src.evaluate import evaluate
from src.model import UNet

# ---------- 1. loss function (same for every fold) ----------
freq = pd.read_csv(EDA_CLASS_DIST, index_col="split").loc["train"]
freq = torch.tensor(freq.values, dtype=torch.float32)
weights = (freq.median() / freq) ** WEIGHT_EXPONENT #Rare classes get bigger weights so mistakes on them cost more, the square root softens weights that were too agressive in an earlier run

ce = nn.CrossEntropyLoss(weight=weights.to(device), ignore_index=IGNORE_INDEX)
lovasz = LovaszLoss(mode="multiclass", from_logits=True, ignore_index=IGNORE_INDEX)

def criterion(logits, masks): # Combines cross-entropy with Lovasz the same loss as the group model
    return CE_COEFF * ce(logits, masks) + LOVASZ_COEFF * lovasz(logits, masks)

# ---------- 2. one training epoch ----------
def train_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0.0
    for imgs, masks in tqdm(loader, leave=False):
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True).long()

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp): #Mixed precision: faster and uses less GPU memory 
            logits = model(imgs)
            loss = criterion(logits, masks)

        optimizer.zero_grad() #The learning steps: clear old gradients, calculate new ones from the loss, update the weights 
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    scheduler.step() #Lowers the learning rate once per epoch along the cosine curve 
    return total_loss / len(loader)

# ---------- 3. the fold loop ----------
results = []

for fold in FOLDS:
    fold_out = MODEL_DIR / f"fold_{fold}"
    fold_out.mkdir(parents=True, exist_ok=True)
    done_file = fold_out / "best.json"

    if done_file.exists(): # Skips folds that already finished so a restart doesnt redo them 
        print(f"Fold {fold} already finished, skipping")
        with open(done_file) as f:
            results.append(json.load(f))
        continue

    print(f"\n===== Fold {fold}: validating on {FOLDS[fold]} =====")

    torch.manual_seed(42) #Every fold starts from the same random weights so the differences come from the data 
    model = UNet(in_channels=3, num_classes=NUM_CLASSES, base=BASE).to(device) #Each fold starts completely from scratch so info from one fold doesnt leak into the next folds validation
    optimizer = torch.optim.AdamW(model.parameters(), lr=MAX_LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LEARNING_RATE)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    train_loader, valid_loader = make_loaders(CV_DIR / f"fold_{fold}")

    history = []
    best = None
    start = time.time()

    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, scaler)
        metrics = evaluate(model, valid_loader, device, num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX, criterion=criterion)#Measures the model on this folds valiation cities after every epocj

        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": float(metrics["loss"]),
            "pixel_accuracy": float(metrics["pixel_accuracy"]),
            "mean_iou": float(metrics["mean_iou_class"]),
            "mean_class_accuracy": float(metrics["mean_class_accuracy"]),
        }
        for name, iou, acc in zip(metrics["class_names"], metrics["iou_per_class"], metrics["accuracy_per_class"]):
            column = name.replace(" ", "_")
            row[f"iou_{column}"] = float(iou)
            row[f"accuracy_{column}"] = float(acc)

        history.append(row)
        pd.DataFrame(history).to_csv(fold_out / "history.csv", index=False)

        if best is None or row["mean_iou"] > best["mean_iou"]: #keeps the best epoch by validation cities after every epoch
            best = {"fold": fold, "best_epoch": epoch + 1,
                    "mean_iou": row["mean_iou"],
                    "pixel_accuracy": row["pixel_accuracy"],
                    "mean_class_accuracy": row["mean_class_accuracy"],
                    "val_loss": row["val_loss"]}
            torch.save(model.state_dict(), fold_out / "model.pt")

        print(f"Fold {fold} | Epoch {epoch + 1:02d}/{EPOCHS} | "
              f"train loss: {train_loss:.4f} | val loss: {row['val_loss']:.4f} | "
              f"mIoU: {row['mean_iou']:.4f} | pixel acc: {row['pixel_accuracy']:.4f}")

    best["minutes"] = round((time.time() - start) / 60, 1)
    with open(done_file, "w") as f:
        json.dump(best, f, indent=2)
    results.append(best)

    # free GPU memory before the next fold
    del model, optimizer, scheduler, scaler, train_loader, valid_loader 
    torch.cuda.empty_cache()

print("\nAll folds finished")

In [ ]:
#Combines the 5 folds best results into 1 table. With mean and standard devation.
import json
import pandas as pd
# Reads each folds best.json from disk instead of using variables in memory
rows = []
for fold in FOLDS:
    with open(MODEL_DIR / f"fold_{fold}" / "best.json") as f:
        rows.append(json.load(f))

results_df = pd.DataFrame(rows).set_index("fold") #turns the 5 dictionaries into a table

metrics = ["mean_iou", "pixel_accuracy", "mean_class_accuracy", "val_loss"] #The columns to summarise the data
summary = results_df[metrics].agg(["mean", "std"])

table = pd.concat([results_df[["best_epoch"] + metrics + ["minutes"]], summary])
display(table.round(4))

table.to_csv(MODEL_DIR / "cv_results.csv") # Saves the table so it can be opened in Excel and used in report 
print("Saved to", MODEL_DIR / "cv_results.csv")

In [ ]:
#Plots ever folds trainig curves on one chart 
import matplotlib.pyplot as plt

#One figure with 2 charts side by side 
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

#One line per fold, so all 5 can be compared on the same chart
for fold in FOLDS:
    hist = pd.read_csv(MODEL_DIR / f"fold_{fold}" / "history.csv") #Reads that fold’s saved per-epoch results
    axes[0].plot(hist["epoch"], hist["mean_iou"], label=f"Fold {fold}")#mIoU is the main Cityscapes metric, so it’s the most important curve
    axes[1].plot(hist["epoch"], hist["val_loss"], label=f"Fold {fold}")#Validation loss is plotted to check for overfitting

axes[0].set_title("Validation mIoU per Fold")#Titles, labels, legend and grid so the figure can go straight into the report
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("mIoU")
axes[0].legend()
axes[0].grid(True)

axes[1].set_title("Validation Loss per Fold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(MODEL_DIR / "cv_curves.png", dpi=200, bbox_inches="tight") #Saves the graph as a png for the report 
plt.show()